# Task 1 – MRI-Only Diagnostic Decision (Phase 3, revised)

**Goal (from the challenge):** from MRI information + clinical data, recommend **biopsy: "yes" / "no"** *before* histopathology is known, and explain the decision.

**Ground truth:** csPCa (ISUP ≥ 2) confirmed by histopathology; negatives confirmed by PSA follow-up. In our data `label = 1` means csPCa, i.e. **"yes"**.

**Output per case: two JSON files**
- `<case_id>_decision.json` → the binary recommendation
- `<case_id>_reasoning.json` → confidence, importance of 10 clinical variables, sections accessed, free-text explanation

## How the system works
```
 clinical + MRI data ─► ENSEMBLE (LR + RF + GB [+ MRI branch]) ─► risk score ─► "yes"/"no"
                              │
                              └─► REASONING LAYER (built on the model's own behaviour)
                                    • confidence        ← distance from threshold + model disagreement
                                    • variable importance ← "what happens to the risk if we neutralise this variable?"
                                    • sections accessed  ← same test, applied to whole report sections
                                    • explanation        ← sentence template filled with the patient's real values
```
**Honesty note:** a statistical model does not literally "open" sections like an agent. The *sections accessed* field therefore means *sections whose information measurably changed the risk score*. This is stated in the write-up and in the JSON documentation.

**Important rule:** `reasoning_audit.csv` (the reference reasoning) is **never used for training or to tune the reasoning rules**. It is only used at the end to *evaluate* how close our reasoning is to the reference.

## 1. Load data

**Why:** we need (a) the cleaned features and MRI embeddings from Phase 1, (b) the raw `features.csv` only to show human-readable values (e.g. "PSA 4.7") in explanations, and (c) the reference reasoning for the final evaluation.

In [1]:
import json, warnings, os, zipfile
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, cohen_kappa_score
warnings.filterwarnings('ignore')
pd.set_option('display.width', 160)

DATA_DIR = '.'
clean = pd.read_csv(f'{DATA_DIR}/features_clean.csv')
emb   = pd.read_csv(f'{DATA_DIR}/embeddings_aligned.csv')
meta  = json.load(open(f'{DATA_DIR}/feature_lists.json'))
raw   = pd.read_csv(f'{DATA_DIR}/features.csv').set_index('case_id')          # only for readable values
audit = pd.read_csv(f'{DATA_DIR}/reasoning_audit.csv').set_index('case_id')   # EVALUATION ONLY

RARE = ['flag_anticoagulant', 'dre_not_done', 'zone_unknown']                # too few positives to learn from
FEATURES = [c for c in meta['features'] if c not in RARE]
EMB_COLS = meta['embedding_columns']

X_full = pd.concat([clean[FEATURES], emb[EMB_COLS]], axis=1)                  # clinical + MRI representation
case_ids = clean['case_id']
is_lab = clean['label'].notna().values

X, y, ids = X_full[is_lab].reset_index(drop=True), clean.loc[is_lab, 'label'].astype(int).reset_index(drop=True), case_ids[is_lab].reset_index(drop=True)
X_new, ids_new = X_full[~is_lab].reset_index(drop=True), case_ids[~is_lab].reset_index(drop=True)
print(f'labelled: {len(y)} | unlabelled (to predict): {len(X_new)} | clinical features: {len(FEATURES)} | MRI embedding dims: {len(EMB_COLS)}')

labelled: 91 | unlabelled (to predict): 104 | clinical features: 56 | MRI embedding dims: 1024


## 2. Map features to the 10 clinical variables and 4 report sections

**Why:** the required reasoning is expressed per *clinical variable* (psa, psad, vol, pirads, dre, fh, comorbidity, cspca, age, bx) and per *section* (radiology_report, previous_notes, psa_trend, laboratory_results). Our model uses 56 columns, so we first say which columns belong to which variable/section. Columns not listed (e.g. vitals, BMI) still feed the model but are not reported.

In [2]:
VAR_GROUPS = {
    'psa':         ['psa_log', 'psa_first_log', 'psa_max_log', 'psap_log', 'psav_log',
                    'psa_slope_per_step_log', 'psa_last_over_first_log', 'psa_trend_n'],
    'psad':        ['psad_log'],
    'vol':         ['vol_log'],
    'pirads':      ['pirads', 'lesion_max_mm', 'lesion_max_mm_missing', 'lesion_mm_report',
                    'extracapsular_ext', 'zone_non_periph'],
    'dre':         ['dre_abnormal'],
    'fh':          ['fh'],
    'comorbidity': ['n_comorbidities', 'flag_alpha_blocker'],
    'cspca':       ['cspca'],
    'age':         ['age'],
    'bx':          ['bx_cat_Negative', 'bx_cat_None', 'bx_cat_Positive', 'has_prior_isup', 'prior_isup_grade',
                    'active_surveillance', 'enc_type_Follow-up - PSA + MRI',
                    'enc_type_Re-evaluation - prior PCa diagnosis', 'enc_type_Re-evaluation - prior negative biopsy'],
}
SEC_GROUPS = {
    'radiology_report':   VAR_GROUPS['pirads'] + VAR_GROUPS['vol'],
    'previous_notes':     VAR_GROUPS['bx'] + ['n_previous_notes'],
    'psa_trend':          [c for c in VAR_GROUPS['psa'] if c != 'psa_log'],
    'laboratory_results': [c for c in FEATURES if c.startswith('lab_')],
}
for name, cols in {**VAR_GROUPS, **SEC_GROUPS}.items():
    missing = [c for c in cols if c not in FEATURES]
    assert not missing, (name, missing)
print('mapping OK:', len(VAR_GROUPS), 'variables,', len(SEC_GROUPS), 'sections')

mapping OK: 10 variables, 4 sections


## 3. Models and the ensemble (+ optional MRI branch)

**Why this ensemble (from Phase 3):** Logistic Regression (straight-line rule), Random Forest and Gradient Boosting (shallow trees, catch interactions) make different mistakes; averaging their probabilities is steadier than any one. Settings are deliberately cautious because we only have 91 labelled patients.

**Why an MRI branch?** The challenge says to use the MRI neural representation. 1024 numbers for 89 patients is too many, so the branch first shrinks them to 5 components (PCA), then runs a strongly regularised logistic regression. It is added as a **4th member** and **kept only if cross-validation shows it helps** (section 4).

**Why `ColumnTransformer` at the start of each member?** Each member picks only its own columns (clinical members ignore the embeddings; the MRI member ignores the clinical columns), while the ensemble receives one table.

In [3]:
def clinical_pipe(model, scale):
    steps = [('select', ColumnTransformer([('keep', 'passthrough', FEATURES)], remainder='drop')),
             ('impute', SimpleImputer(strategy='median'))]
    if scale: steps.append(('scale', StandardScaler()))
    return Pipeline(steps + [('model', model)])

def lr_member(): return clinical_pipe(LogisticRegression(C=0.01, max_iter=5000), scale=True)
def rf_member(): return clinical_pipe(RandomForestClassifier(n_estimators=300, max_depth=3, min_samples_leaf=5,
                                                            max_features='sqrt', random_state=0, n_jobs=-1), scale=False)
def gb_member(): return clinical_pipe(GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=2,
                                                                subsample=0.7, min_samples_leaf=5, random_state=0), scale=False)
def mri_member():
    mri_prep = ColumnTransformer([('mri', Pipeline([('impute', SimpleImputer(strategy='median')),
                                                    ('scale', StandardScaler()),
                                                    ('pca', PCA(n_components=5, random_state=0))]), EMB_COLS)],
                                 remainder='drop')
    return Pipeline([('prep', mri_prep), ('model', LogisticRegression(C=0.01, max_iter=5000))])

def make_ensemble(use_mri):
    members = [('lr', lr_member()), ('rf', rf_member()), ('gb', gb_member())]
    if use_mri: members.append(('mri', mri_member()))
    return VotingClassifier(members, voting='soft')       # equal-weight average of probabilities

## 4. Do the MRI embeddings help? (ablation on identical CV splits)

**Why:** adding a model that is mostly noise can make the ensemble worse. We compare *with* and *without* the MRI branch on the **same 50 train/test splits** (repeated stratified 5-fold) and keep it only if the mean AUC improves by at least 0.01. This is decided by the data, not by us.

In [4]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
auc_no  = cross_validate(make_ensemble(False), X, y, cv=cv, scoring='roc_auc')['test_score']
auc_yes = cross_validate(make_ensemble(True),  X, y, cv=cv, scoring='roc_auc')['test_score']
print(f'Ensemble WITHOUT MRI branch: AUC = {auc_no.mean():.3f} (± {auc_no.std():.3f})')
print(f'Ensemble WITH    MRI branch: AUC = {auc_yes.mean():.3f} (± {auc_yes.std():.3f})')
print(f'gain = {auc_yes.mean()-auc_no.mean():+.3f}, MRI version wins in {100*(auc_yes>auc_no).mean():.0f}% of splits')

USE_MRI = bool(auc_yes.mean() - auc_no.mean() >= 0.01)
print('\nUse MRI branch in the final system:', USE_MRI)
if not USE_MRI:
    print('MRI information still enters through PI-RADS, lesion size/zone and extracapsular extension from the radiology report.')

# the MRI embeddings become a reportable group only if they are in the model
if USE_MRI:
    VAR_GROUPS['mri_embedding'] = EMB_COLS
    SEC_GROUPS['mri_representation'] = EMB_COLS

Ensemble WITHOUT MRI branch: AUC = 0.732 (± 0.108)
Ensemble WITH    MRI branch: AUC = 0.699 (± 0.115)
gain = -0.032, MRI version wins in 6% of splits

Use MRI branch in the final system: False
MRI information still enters through PI-RADS, lesion size/zone and extracapsular extension from the radiology report.


## 5. Reasoning layer – the tools

All of these work on **any fitted ensemble**, so we can reuse them for cross-validation and for the final model.

**Variable importance by "neutralising":** for one patient, replace a variable's columns by what a typical patient looks like (the training median) and see how much the risk score moves.
- Big move = the model relied on that variable. The sign says whether it pushed *toward* biopsy (+) or *away* (−).
- It is model-agnostic (works for logistic regression and trees alike), needs no extra library, and is easy to explain.
- Limitation: correlated variables share credit, so treat it as the model's behaviour, not as proof of cause.

**Confidence:** a decision is more trustworthy when (a) the risk score is **far from the threshold** and (b) the member models **agree**.

In [5]:
def neutralise_effect(model, X_batch, baseline, groups):
    # change in risk score caused by each group: risk(full) - risk(group replaced by typical values)
    p_full = model.predict_proba(X_batch)[:, 1]
    effects = {}
    for name, cols in groups.items():
        X_off = X_batch.copy()
        for c in cols:
            X_off[c] = baseline[c]
        effects[name] = p_full - model.predict_proba(X_off)[:, 1]
    return pd.DataFrame(effects)

def analyse(model, X_train, X_batch):
    # everything we need about a batch of patients from one fitted ensemble
    baseline = X_train.median()
    member_p = np.column_stack([m.predict_proba(X_batch)[:, 1] for m in model.estimators_])
    return {'p': model.predict_proba(X_batch)[:, 1],
            'disagree': member_p.std(axis=1),                       # how much the member models disagree
            'var': neutralise_effect(model, X_batch, baseline, VAR_GROUPS),
            'sec': neutralise_effect(model, X_batch, baseline, SEC_GROUPS)}

## 6. Honest per-patient results for the 91 labelled cases (out-of-fold)

**Why:** if we explained labelled patients with a model that was trained on them, the result would look unrealistically good. In 5-fold CV each patient is analysed by a model that **never saw that patient**. We repeat with 5 different shuffles and average, so each patient gets stable numbers.

In [6]:
N_REP = 5
acc = {'p': np.zeros(len(y)), 'disagree': np.zeros(len(y)),
       'var': np.zeros((len(y), len(VAR_GROUPS))), 'sec': np.zeros((len(y), len(SEC_GROUPS)))}

for seed in range(N_REP):
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y):
        model = make_ensemble(USE_MRI).fit(X.iloc[tr], y.iloc[tr])
        r = analyse(model, X.iloc[tr], X.iloc[te])
        acc['p'][te]        += r['p'] / N_REP
        acc['disagree'][te] += r['disagree'] / N_REP
        acc['var'][te]      += r['var'].values / N_REP
        acc['sec'][te]      += r['sec'].values / N_REP

oof = {'p': acc['p'], 'disagree': acc['disagree'],
       'var': pd.DataFrame(acc['var'], columns=list(VAR_GROUPS)),
       'sec': pd.DataFrame(acc['sec'], columns=list(SEC_GROUPS))}
print('done. out-of-fold AUC (pooled over repeats, slightly optimistic): %.3f' % roc_auc_score(y, oof['p']))

done. out-of-fold AUC (pooled over repeats, slightly optimistic): 0.759


## 7. The yes/no decision: choosing the threshold and measuring performance

**Why Youden's J:** the challenge asks for a binary decision, so we need one cut-off. Youden's J (sensitivity + specificity − 1) picks the threshold that balances finding csPCa cases against avoiding unnecessary biopsies. A second, more cautious cut-off (sensitivity ≥ 90%, favouring not missing cancers) is shown for comparison.

**Caveat:** the threshold is chosen on the same out-of-fold scores we report, so the numbers are a little optimistic. The honest headline is the fold-level AUC from section 4.

In [7]:
fpr, tpr, thr = roc_curve(y, oof['p'])
THRESHOLD = float(thr[np.argmax(tpr - fpr)])                    # Youden's J
THRESHOLD_SENS90 = float(thr[np.where(tpr >= 0.90)[0][0]])

def report(threshold, name):
    pred = (oof['p'] >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return {'rule': name, 'threshold': round(threshold, 3), 'accuracy': (tp+tn)/len(y),
            'sensitivity': tp/(tp+fn), 'specificity': tn/(tn+fp), 'precision': tp/(tp+fp),
            'F1': 2*tp/(2*tp+fp+fn)}
print(pd.DataFrame([report(THRESHOLD, 'Youden (balanced)  <-- used'), report(THRESHOLD_SENS90, 'sensitivity >= 90%')]).round(3).to_string(index=False))

                       rule  threshold  accuracy  sensitivity  specificity  precision    F1
Youden (balanced)  <-- used      0.639     0.736        0.696        0.800      0.848 0.765
         sensitivity >= 90%      0.488     0.681        0.911        0.314      0.680 0.779


## 8. Confidence: clear / borderline / uncertain

**Why this rule:** a case is *uncertain* when the score sits very close to the threshold or when the member models disagree strongly; *clear* when it is far from the threshold and the models agree.
The cut-offs are set from the **distribution of our own out-of-fold scores** (lowest 20% of distances = uncertain, next 25% = borderline, rest = clear; strong disagreement downgrades one level). They are **not** tuned with the reference file.

**Validity check:** a good confidence label should mean what it says: accuracy should be highest for "clear" and lowest for "uncertain".

In [8]:
margin_oof = np.abs(oof['p'] - THRESHOLD)
CUT_UNCERTAIN, CUT_CLEAR = np.quantile(margin_oof, [0.20, 0.45])
CUT_DISAGREE = np.quantile(oof['disagree'], 0.90)

def confidence_level(p, disagree):
    margin = np.abs(np.asarray(p) - THRESHOLD)
    level = np.where(margin < CUT_UNCERTAIN, 0, np.where(margin < CUT_CLEAR, 1, 2))      # 0 uncertain, 1 borderline, 2 clear
    level = np.where(np.asarray(disagree) > CUT_DISAGREE, np.maximum(level - 1, 0), level)
    return np.array(['uncertain', 'borderline', 'clear'])[level]

conf_oof = confidence_level(oof['p'], oof['disagree'])
correct = ((oof['p'] >= THRESHOLD).astype(int) == y.values)
print(pd.DataFrame({'confidence': conf_oof, 'correct': correct}).groupby('confidence')['correct']
        .agg(cases='size', accuracy='mean').reindex(['clear', 'borderline', 'uncertain']).round(2))

            cases  accuracy
confidence                 
clear          43      0.79
borderline     29      0.66
uncertain      19      0.74


## 9. Importance levels and sections accessed

**Why fixed rules:** we convert the numeric effects into the same vocabulary as the reference (decisive / important / noted / not_used):
- compare each variable's |effect| with the **largest effect in that patient**: ≥ 75% → *decisive*, ≥ 40% → *important*, ≥ 10% → *noted*, below → *not_used*;
- an effect smaller than 0.5 percentage points of risk is always *not_used* (it is noise, however large it looks relatively).

**Note:** these levels describe what *this model* relied on. If the model concentrates on a few variables, the others will correctly show as *not_used*, even where a clinician would have looked at them.

**Sections accessed:** a section counts as accessed if neutralising it changes the risk by at least 1 percentage point; they are listed from most to least influential (at least one is always listed).

In [9]:
LEVEL_NAMES = ['not_used', 'noted', 'important', 'decisive']
ABS_FLOOR = 0.005          # 0.5 percentage points of risk

def importance_levels(effects_row):
    a = np.abs(effects_row.values.astype(float))
    rel = a / a.max() if a.max() > 0 else a
    lvl = np.where(rel >= 0.75, 3, np.where(rel >= 0.40, 2, np.where(rel >= 0.10, 1, 0)))
    lvl = np.where(a < ABS_FLOOR, 0, lvl)
    return {k: LEVEL_NAMES[i] for k, i in zip(effects_row.index, lvl)}

def sections_accessed(effects_row, min_effect=0.01):
    a = effects_row.abs().sort_values(ascending=False)
    keep = a[a >= min_effect].index.tolist()
    return keep if keep else [a.index[0]]

## 10. Free-text explanation (template)

**Why a template instead of free generation:** every sentence is built from the patient's *actual* values and the model's *actual* effects, so it cannot invent facts. It names the factors that support the decision, the strongest factor against it, and flags weak evidence.

In [10]:
def describe(var, cid, idx_clean):
    r, c = raw.loc[cid], clean.iloc[idx_clean]
    f = {
        'pirads': lambda: f"PI-RADS {int(r['pirads'])}" if pd.notna(r['pirads']) else 'PI-RADS not available',
        'psad':   lambda: f"PSA density {r['psad']:.2f}",
        'psa':    lambda: f"PSA {r['psa']:.1f} ng/mL" + (f" (velocity {r['psav']:+.1f})" if pd.notna(r['psav']) else ''),
        'vol':    lambda: f"prostate volume {r['vol']:.0f} mL",
        'dre':    lambda: f"DRE: {str(r['dre']).lower()}",
        'age':    lambda: f"age {int(r['age'])}",
        'fh':     lambda: 'family history of prostate cancer' if r['fh'] == 1 else 'no family history',
        'comorbidity': lambda: f"{int(r['n_comorbidities'])} comorbidities",
        'cspca':  lambda: f"csPCa likelihood score {r['cspca']:.2f}" if pd.notna(r['cspca']) else 'csPCa score not available',
        'bx':     lambda: ('no prior biopsy' if pd.isna(r['bx']) else f"prior biopsy {str(r['bx']).lower()}")
                          + (f", prior ISUP {int(c['prior_isup_grade'])}" if c['prior_isup_grade'] > 0 else ''),
        'mri_embedding': lambda: 'MRI neural representation',
    }
    return f[var]()

def explanation_text(cid, idx_clean, p, conf, var_effects, secs):
    yes = p >= THRESHOLD
    sign = 1 if yes else -1
    eff = var_effects * sign                                    # positive = supports the decision
    support = eff[eff > ABS_FLOOR].sort_values(ascending=False).index[:3].tolist()
    against = eff[eff < -ABS_FLOOR].sort_values().index[:1].tolist()
    head = f"Biopsy {'recommended' if yes else 'not recommended'} (risk score {p:.2f} {'>=' if yes else '<'} decision threshold {THRESHOLD:.2f}; confidence: {conf})."
    parts = [head]
    if support: parts.append('Main supporting factors: ' + '; '.join(describe(v, cid, idx_clean) for v in support) + '.')
    if against: parts.append('Strongest factor pointing the other way: ' + describe(against[0], cid, idx_clean) + '.')
    if conf == 'borderline': parts.append('Evidence is mixed, so the score is close to the decision threshold.')
    if conf == 'uncertain':  parts.append('The score is very close to the threshold or the models disagree; clinical review is advised.')
    parts.append('Sections informing this decision: ' + ', '.join(secs) + '.')
    return ' '.join(parts)

## 11. Assemble the outputs for all 195 cases

**Why two sources:** for the **104 unlabelled** patients we use the **final model trained on all 91 labelled cases** (a genuine prediction). For the **91 labelled** patients we use the **out-of-fold** results from section 6, because a model that was trained on a patient would give a misleadingly confident answer for that same patient.

In [11]:
final_model = make_ensemble(USE_MRI).fit(X, y)
res_new = analyse(final_model, X, X_new)

def build_records(case_ids_, idx_in_clean, p, disagree, var_df, sec_df, source):
    conf = confidence_level(p, disagree)
    recs = []
    for i, cid in enumerate(case_ids_):
        levels = importance_levels(var_df.iloc[i])
        secs = sections_accessed(sec_df.iloc[i])
        recs.append({
            'case_id': cid, 'source': source, 'risk_score': float(p[i]),
            'decision': {'case_id': cid, 'biopsy_recommendation': 'yes' if p[i] >= THRESHOLD else 'no'},
            'reasoning': {'case_id': cid, 'confidence': conf[i], 'variable_importance': levels,
                          'sections_accessed': secs,
                          'explanation': explanation_text(cid, idx_in_clean[i], p[i], conf[i], var_df.iloc[i], secs)},
        })
    return recs

idx_lab = np.where(is_lab)[0]; idx_new = np.where(~is_lab)[0]
records  = build_records(ids,     idx_lab, oof['p'],     oof['disagree'],     oof['var'],     oof['sec'],     'labelled (out-of-fold)')
records += build_records(ids_new, idx_new, res_new['p'], res_new['disagree'], res_new['var'], res_new['sec'], 'unlabelled (final model)')
print(len(records), 'cases assembled\n')
print(json.dumps(records[0]['reasoning'], indent=2))

195 cases assembled

{
  "case_id": "PT-pseudo_0020cfca66c8",
  "confidence": "borderline",
  "variable_importance": {
    "psa": "not_used",
    "psad": "not_used",
    "vol": "noted",
    "pirads": "decisive",
    "dre": "not_used",
    "fh": "not_used",
    "comorbidity": "not_used",
    "cspca": "noted",
    "age": "not_used",
    "bx": "not_used"
  },
  "sections_accessed": [
    "radiology_report",
    "laboratory_results",
    "psa_trend"
  ],
  "explanation": "Biopsy not recommended (risk score 0.58 < decision threshold 0.64; confidence: borderline). Main supporting factors: PI-RADS 2; csPCa likelihood score 0.60. Strongest factor pointing the other way: prostate volume 33 mL. Evidence is mixed, so the score is close to the decision threshold. Sections informing this decision: radiology_report, laboratory_results, psa_trend."
}


## 12. Write the JSON files and validate them

**Why validate:** the challenge expects a fixed structure, so before saving we check every file: required keys, allowed values (`yes`/`no`; clear/borderline/uncertain; the four importance levels), and size (~5 KB at most).

In [12]:
OUT_DIR = 'task1_outputs'; os.makedirs(OUT_DIR, exist_ok=True)
VAR_KEYS = set(VAR_GROUPS)

def validate(rec):
    d, r = rec['decision'], rec['reasoning']
    assert set(d) == {'case_id', 'biopsy_recommendation'} and d['biopsy_recommendation'] in ('yes', 'no')
    assert set(r) == {'case_id', 'confidence', 'variable_importance', 'sections_accessed', 'explanation'}
    assert r['confidence'] in ('clear', 'borderline', 'uncertain')
    assert set(r['variable_importance']) == VAR_KEYS and set(r['variable_importance'].values()) <= set(LEVEL_NAMES)
    assert len(r['sections_accessed']) >= 1 and len(r['explanation']) > 20

max_bytes = 0
for rec in records:
    validate(rec)
    for kind in ('decision', 'reasoning'):
        text = json.dumps(rec[kind], indent=2)
        max_bytes = max(max_bytes, len(text.encode()))
        open(f"{OUT_DIR}/{rec['case_id']}_{kind}.json", 'w').write(text)

summary = pd.DataFrame([{'case_id': r['case_id'], 'source': r['source'], 'risk_score': round(r['risk_score'], 3),
                         'biopsy_recommendation': r['decision']['biopsy_recommendation'],
                         'confidence': r['reasoning']['confidence']} for r in records])
summary['true_label'] = summary['case_id'].map(dict(zip(clean['case_id'], clean['label'])))
summary.to_csv('task1_summary.csv', index=False)

with zipfile.ZipFile('task1_outputs.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT_DIR)): z.write(f'{OUT_DIR}/{f}', f)
print(f'wrote {2*len(records)} JSON files, all valid; largest file = {max_bytes} bytes')
print(summary.groupby('source')['biopsy_recommendation'].value_counts().unstack())

wrote 390 JSON files, all valid; largest file = 933 bytes
biopsy_recommendation     no  yes
source                           
labelled (out-of-fold)    45   46
unlabelled (final model)  62   42


## 13. How close is our reasoning to the reference? (evaluation only)

**Why:** `reasoning_audit.csv` holds the reference reasoning for the 91 labelled cases. We compare our **out-of-fold** reasoning with it. Agreement is reported with metrics that account for chance; with 91 cases and a small model, expect **modest** agreement and read the numbers as indications, not as proof.

- **Confidence:** linear weighted Cohen's κ (uncertain < borderline < clear; 0 = chance, 1 = perfect).
- **Variable importance:** per variable, weighted κ and Spearman correlation on the 4 ordered levels.
- **Sections:** Jaccard overlap between the two sets of sections.

In [13]:
ref = audit.reindex(ids.values)
ours = {r['case_id']: r['reasoning'] for r in records if r['source'].startswith('labelled')}

# --- confidence ---
order_c = {'uncertain': 0, 'borderline': 1, 'clear': 2}
c_ref = ref['confidence'].map(order_c); c_our = pd.Series([order_c[ours[c]['confidence']] for c in ids], index=ref.index)
ok = c_ref.notna()
print('CONFIDENCE  weighted kappa = %.2f   exact agreement = %.0f%%' %
      (cohen_kappa_score(c_ref[ok], c_our[ok], weights='linear'), 100*(c_ref[ok] == c_our[ok]).mean()))
print(pd.crosstab(ref['confidence'], pd.Series([ours[c]['confidence'] for c in ids], index=ref.index),
                  rownames=['reference'], colnames=['ours']).reindex(index=['clear', 'borderline', 'uncertain'],
                  columns=['clear', 'borderline', 'uncertain'], fill_value=0))

# --- variable importance ---
order_i = {k: i for i, k in enumerate(LEVEL_NAMES)}
rows = []
for v in [v for v in VAR_GROUPS if f'w_{v}' in ref.columns]:
    a = ref[f'w_{v}'].map(order_i); b = pd.Series([order_i[ours[c]['variable_importance'][v]] for c in ids], index=ref.index)
    m = a.notna()
    rows.append({'variable': v, 'ref mean level': a[m].mean(), 'our mean level': b[m].mean(),
                 'spearman': spearmanr(a[m], b[m])[0], 'weighted kappa': cohen_kappa_score(a[m], b[m], weights='linear', labels=[0, 1, 2, 3])})
imp_tbl = pd.DataFrame(rows).set_index('variable').round(2)
print('\nVARIABLE IMPORTANCE (levels: 0 not_used ... 3 decisive)'); print(imp_tbl)
print('average weighted kappa over variables: %.2f' % imp_tbl['weighted kappa'].mean())

# --- sections ---
jac = []
for c in ids:
    rs = ref.loc[c, 'reveal_sequence']
    if pd.isna(rs): continue
    A, B = set(rs.split(';')), set(ours[c]['sections_accessed']) & set(SEC_GROUPS)
    jac.append(len(A & B) / len(A | B))
print('\nSECTIONS  mean Jaccard overlap = %.2f  (over %d cases)' % (np.mean(jac), len(jac)))

CONFIDENCE  weighted kappa = 0.17   exact agreement = 47%
ours        clear  borderline  uncertain
reference                               
clear          31          19          8
borderline      7           6          5
uncertain       5           4          6

VARIABLE IMPORTANCE (levels: 0 not_used ... 3 decisive)
             ref mean level  our mean level  spearman  weighted kappa
variable                                                             
psa                    1.77            1.58      0.10            0.06
psad                   1.29            0.19      0.01            0.01
vol                    1.18            0.87      0.19            0.05
pirads                 2.47            0.77     -0.05           -0.03
dre                    1.26            0.07      0.23            0.00
fh                     0.77            0.00       NaN            0.00
comorbidity            0.85            0.11      0.30            0.06
cspca                  0.46            0.37     -0

In [14]:
# --- does our explanation mention the same key factors as the reference free text? (rough keyword check) ---
def mentions(text, words): return any(w in str(text).lower() for w in words)
for name, words in {'PSA': ['psa'], 'PI-RADS': ['pirads', 'pi-rads', 'pi rads', 'priads']}.items():
    r_has = np.array([mentions(ref.loc[c, 'free_text'], words) for c in ids])
    o_has = np.array([mentions(ours[c]['explanation'], words) for c in ids])
    print(f'{name:8s} reference mentions it in {r_has.sum():2d} cases; ours also mentions it in {(r_has & o_has).sum():2d} of those')

PSA      reference mentions it in 33 cases; ours also mentions it in 29 of those
PI-RADS  reference mentions it in 29 cases; ours also mentions it in 17 of those


## 14. What to report

1. **Decision quality:** fold-level AUC (section 4), plus accuracy / sensitivity / specificity at the chosen threshold (section 7) – with the caveat that the threshold was chosen on the same scores.
2. **MRI representation:** the ablation in section 4. If the embeddings did not help, say so and explain that MRI information still enters through PI-RADS and report-derived lesion features.
3. **Reasoning quality:** the confidence validity check (section 8) and the agreement with the reference (section 13), with the small-sample caveat.
4. **Limitations:** 91 labelled cases; importance and sections are derived from model behaviour (not an agent log); the explanation is a template.
5. **Deliverables:** `task1_outputs/` (or `task1_outputs.zip`), 2 JSON files per case, and `task1_summary.csv`.